In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

from _load_brain_atlases import load_brain_atlases
from _results_computation import (read_pickle, compare_conditions, threshold_schaefers,
                                  find_max, save_schaefer_maps, plot_shared_voxels)

In [ ]:
# load in records
# comparing recursion and iteration
records = read_pickle("output/organized_results_by_condition.pkl")
nulls = read_pickle('output/null_distribution_condition_intersections.pkl')
atlases = load_brain_atlases()

In [3]:
sig_voxels = {
    'recursion':[],
    'iteration':[],
    'prose'    :[],
    'list'     :[]
}

corr_coefficients = {
    'recursion':[],
    'iteration':[],
    'prose'    :[],
    'list'     :[]
}

all_sigs = []
for i,row in records.iterrows():
    # initially saved the voxels that weren't significant by mistake, so calculating the number of voxels that are
    num_sig_voxels = len(row['sig_voxel_idx'])
    condition = row['condition']
    sig_voxels[condition].append(num_sig_voxels)
    all_sigs.append(num_sig_voxels)

    corr_coefficients[condition].append(row['participant_mean'])

    
for condition,vox in sig_voxels.items():
    print(f"Average number of significant voxels in {condition}: {np.mean(vox):.3f} ($\\sigma={np.std(vox):.3f}$). Average correlation is {np.mean(corr_coefficients[condition])} ($\\sigma={np.std(corr_coefficients[condition])}$)")
print(f"Average number of significant voxels across conditions: {np.mean(all_sigs):.3f} (\\signam{np.std(all_sigs):.3f})")

cutoff = round(np.mean(all_sigs))

Average number of significant voxels in recursion: 5269.296 ($\sigma=3116.940$). Average correlation is 0.34019899728088526 ($\sigma=0.060721391063022406$)
Average number of significant voxels in iteration: 5488.741 ($\sigma=3212.501$). Average correlation is 0.387729002328255 ($\sigma=0.09557044321514128$)
Average number of significant voxels in prose: 6265.444 ($\sigma=4563.263$). Average correlation is 0.4170074793989078 ($\sigma=0.14006262987049722$)
Average number of significant voxels in list: 9394.042 ($\sigma=6667.056$). Average correlation is 0.5688592658395214 ($\sigma=0.2544265805212107$)
Average number of significant voxels across conditions: 6524.676 (\signam4821.499)


In [ ]:
# Shared / unique top voxels between recursion and iteration for every participant.
# For every participant, there's a higher intersection than would be expected by chance
# between recursion and iteration conditions
results = compare_conditions(records, nulls, 'recursion', 'iteration')

participant_ids = results['participant_ids']
observed_vals   = results['observed_vals']
null_vals_list  = results['null_vals_list']
pvals           = results['pvals']
schaefers       = results['schaefers']
key_by_subset   = results['key_by_subset']
print(f"{len(participant_ids)} participants")

In [ ]:
# correct for multiple comparisons
reject, qvals, _, _ = multipletests(np.nan_to_num(pvals, nan=1.0), alpha=0.001, method='fdr_bh')
print(reject)

In [ ]:
plot_shared_voxels(participant_ids, observed_vals, null_vals_list,
                   title='Top-modeled Voxels Shared Between Recursion and Iteration')

In [ ]:
count_max = find_max(schaefers)
save_schaefer_maps(schaefers, atlases, vmax=count_max)

In [ ]:
# Group-level significance for the schaefer parcel tallies
# (see threshold_schaefers in _results_computation.py)
schaefers_thresholded, schaefer_qvals = threshold_schaefers(schaefers, nulls, participant_ids, key_by_subset, alpha=0.05)

for subset, qvals in schaefer_qvals.items():
    sig_parcels = np.where(qvals < 0.05)[0]
    print(f"{subset}: {len(sig_parcels)} parcels survive q<0.05 -> {sorted(sig_parcels.tolist())}")

In [ ]:
count_max = find_max(schaefers_thresholded)
save_schaefer_maps(schaefers_thresholded, atlases, vmax=count_max)

In [ ]:
# groups
# recursion similar in prose and code:
# P101, P102, P104, P107, P108, P110, P113, P116, P121, P122, P124, P125
similar = ['102', '104', '107', '108', '110', '113', '116', '121', '122', '124', '125']

# recursion different 
# P101, P105, P109, P112, P113, P114, P115, P118, P120, P124, P125, P127, P130, P131, P203
different = ['105', '109', '112', '113', '114', '115', '118', '120', '124', '125', '127', '130', '131', '203']

# visualization
# P102, P107, P108, P115, P118, P120, P121, P124, P127, P130, P203
visualization = ['102', '107', '108', '115', '118', '120', '121', '124', '127', '130', '203']

# planning 
# P107, P110, P112, P113, P114, P116, P122, P125, P133
planning = ['107', '110', '112', '113', '114', '116', '122', '125', '133']

In [ ]:
for pid,mini_df in records.groupby(['participant']):
    pid = pid[0]
    ann = ''
    if pid in similar:
        ann += ' similar'
    if pid in different:
        ann += ' different'
    if pid in visualization:
        ann += ' visual'
    if pid in planning:
        ann += ' plan'
    # if pid in 
    v = mini_df.set_index('condition')['top_voxel_idx']
    rec_it_intersect = set.intersection(v['recursion'], v['iteration'])
    rec_pro_intersect = set.intersection(v['recursion'], v['prose'])
    print(f"Participant {pid+ann}: Recursion/Iteration intersection {len(rec_it_intersect)} | Recursion/Prose intersection {len(rec_pro_intersect)}")